# Batch runner for SERGIO simulations (gCRL)
This notebook uses the package APIs:

- `gcrl.simulation.grn.build_grn`
- `gcrl.simulation.sergio_sim.run_sergio_sim`

Outputs are saved under `gCRL/data/simulated/` with subfolders reflecting grid settings.

In [8]:
# If you haven't already installed the package in editable mode, run:
# !pip install -e .

import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
from datetime import datetime

from gcrl.simulation.grn import build_grn
from gcrl.simulation.sergio_sim import run_sergio_sim
from gcrl.config import resolve_sergio_dir

print('SERGIO resolved at:', resolve_sergio_dir(None))


/home/laganiv/Desktop/projects/CausalEmbed/grn_crl/gCRL/gcrl.toml
SERGIO resolved at: /home/laganiv/Desktop/projects/CausalEmbed/grn_crl/gCRL/ext_tools/SERGIO


## Output layout

In [9]:
# Root directory relative to the repo (change if needed)
ROOT = Path('../../data/simulated').resolve()
ROOT.mkdir(parents=True, exist_ok=True)
print('Output root:', ROOT)


Output root: /home/laganiv/Desktop/projects/CausalEmbed/grn_crl/gCRL/notebooks/05_simulation/data/simulated


## Parameter sweeps
Define combinations for GRN generation and SERGIO runs.

In [10]:
# Random seeds and size settings
seeds = [12345] #[0, 1]                # expand as needed
n_communities_list = [5] #[3, 5]
n_targets_list = [800] # [400, 800]
n_bins = [1] #3

# SERGIO sizes per dataset
n_unperturbed = 1000
n_perturbed_per_comm = 400

# Fixed SERGIO parameters (can be adjusted)
sergio_kwargs = dict(
    noise_params=1.0,
    decays=0.8,
    sampling_state=15,
    noise_type='dpd',
)

grid = []
for seed in seeds:
    for nc in n_communities_list:
        for nt in n_targets_list:
            grid.append(dict(seed=seed, n_communities=nc, n_targets=nt, n_bins=n_bins))

len(grid), grid[:3]
#grid

(1, [{'seed': 12345, 'n_communities': 5, 'n_targets': 800, 'n_bins': [1]}])

## Run
For each configuration:
1. Build a GRN (writes SERGIO inputs).
2. Run SERGIO to produce a `.h5ad` file.
3. Record run metadata into a summary table.

Each configuration is saved to a subfolder:
`C{nc}_TG{nt}_B{n_bins}_U{n_unpert}_P{n_pert}_S{seed}`

In [11]:
def combo_tag(cfg, n_unperturbed, n_perturbed_per_comm):
    return f"C{cfg['n_communities']}_TG{cfg['n_targets']}_B{cfg['n_bins']}_U{n_unperturbed}_P{n_perturbed_per_comm}_S{cfg['seed']}"

summary_rows = []
for i, cfg in enumerate(grid, 1):
    seed = cfg['seed']
    nc = cfg['n_communities']
    nt = cfg['n_targets']
    nb = cfg['n_bins']

    tag = combo_tag(cfg, n_unperturbed, n_perturbed_per_comm)
    combo_dir = ROOT / tag
    grn_dir = combo_dir / "grn"
    out_h5ad = combo_dir / f"sergio_{tag}.h5ad"
    combo_dir.mkdir(parents=True, exist_ok=True)

    print(f"[{i}/{len(grid)}] → {tag}")
    print(f"  Building GRN → {grn_dir}")
    builder = build_grn(
        outdir=str(grn_dir),
        seed=seed,
        n_communities=nc,
        n_targets=nt,
        n_bins=nb,
        inter_density='five_per_pair',
        p_repression=0.30,
    )

    print(f"  Running SERGIO → {out_h5ad}")
    adata = run_sergio_sim(
        grn_dir=str(grn_dir),
        out_h5ad=str(out_h5ad),
        n_unperturbed=n_unperturbed,
        n_perturbed_per_comm=n_perturbed_per_comm,
        seed=seed + 123,  # separate seed for SERGIO (optional)
        **sergio_kwargs,
    )

    # Basic metadata
    n_cells = adata.n_obs
    n_genes = adata.n_vars
    n_tf = int((adata.var['kind'] == 'TF').sum())
    n_tg = int((adata.var['kind'] == 'TG').sum())
    n_comms = int(pd.Categorical(adata.var['community']).categories.size)

    summary_rows.append(dict(
        tag=tag,
        seed_grn=seed,
        seed_sergio=seed+123,
        n_cells=n_cells,
        n_genes=n_genes,
        n_tf=n_tf,
        n_tg=n_tg,
        n_communities=n_comms,
        n_unperturbed=n_unperturbed,
        n_perturbed_per_comm=n_perturbed_per_comm,
        grn_dir=str(grn_dir),
        out_h5ad=str(out_h5ad),
    ))

summary = pd.DataFrame(summary_rows)
summary_path = ROOT / 'summary.csv'
summary.to_csv(summary_path, index=False)
summary_path, summary.head()


[1/1] → C5_TG800_B[1]_U1000_P400_S12345
  Building GRN → /home/laganiv/Desktop/projects/CausalEmbed/grn_crl/gCRL/notebooks/05_simulation/data/simulated/C5_TG800_B[1]_U1000_P400_S12345/grn
  Running SERGIO → /home/laganiv/Desktop/projects/CausalEmbed/grn_crl/gCRL/notebooks/05_simulation/data/simulated/C5_TG800_B[1]_U1000_P400_S12345/sergio_C5_TG800_B[1]_U1000_P400_S12345.h5ad
/home/laganiv/Desktop/projects/CausalEmbed/grn_crl/gCRL/gcrl.toml/home/laganiv/Desktop/projects/CausalEmbed/grn_crl/gCRL/gcrl.toml/home/laganiv/Desktop/projects/CausalEmbed/grn_crl/gCRL/gcrl.toml/home/laganiv/Desktop/projects/CausalEmbed/grn_crl/gCRL/gcrl.toml/home/laganiv/Desktop/projects/CausalEmbed/grn_crl/gCRL/gcrl.toml/home/laganiv/Desktop/projects/CausalEmbed/grn_crl/gCRL/gcrl.toml





Start simulating new levelStart simulating new level

There are 7 genes to simulate in this layerThere are 7 genes to simulate in this layerStart simulating new level


There are 7 genes to simulate in this layer
Start simulat

(PosixPath('/home/laganiv/Desktop/projects/CausalEmbed/grn_crl/gCRL/notebooks/05_simulation/data/simulated/summary.csv'),
                                tag  seed_grn  seed_sergio  n_cells  n_genes  \
 0  C5_TG800_B[1]_U1000_P400_S12345     12345        12468     3000      887   
 
    n_tf  n_tg  n_communities  n_unperturbed  n_perturbed_per_comm  \
 0    87   800              5           1000                   400   
 
                                              grn_dir  \
 0  /home/laganiv/Desktop/projects/CausalEmbed/grn...   
 
                                             out_h5ad  
 0  /home/laganiv/Desktop/projects/CausalEmbed/grn...  )

## Notes
- You can expand `seeds`, `n_communities_list`, and `n_targets_list`.
- Configure the SERGIO path via `gcrl.toml` at the repo root (recommended) or `GCRL_SERGIO_DIR`.
- For large sweeps, consider parallelizing over the grid.